# Task 2: Forward Kinematics

## 2.1 Rotation in 2D

We need to send create a rotation matrix for each of the two axes (x, y). 

e.g.:
point = np.array([1, 0, 0]) --> rotate around z-axis by 90 degrees (π/2 radians) --> new point = np.array([0, 1, 0])


In [1]:
# ToDo: replace torch with any other autograd library you prefer
import torch
import numpy as np
p0 = torch.tensor([1, 0], dtype=torch.float32)
angle = torch.tensor(np.pi / 2, dtype=torch.float32)  # 90 degrees in radians
# Applied Rotation matrix for 2D rotation
rotation_matrix = torch.tensor([
    [torch.cos(angle), -torch.sin(angle)],
    [torch.sin(angle), torch.cos(angle)]
], dtype=torch.float32)
p_rotated = rotation_matrix @ p0
print("Original point:", p0)
print("Rotated point:", torch.round(p_rotated, decimals=4))
print('Expected output met:', torch.allclose(p_rotated, torch.tensor([0, 1], dtype=torch.float32), atol=1e-4))  # Should be close to [0, 1]

Original point: tensor([1., 0.])
Rotated point: tensor([-0., 1.])
Expected output met: True


Following that, every segment has a some properties:
- origin (the position of the joint)
- rotation (the rotation of the joint (relative to the parent joint, or absolute))
- location in parent (the position of the joint relative to the parent joint)

--> When we recursively apply the rotation and translation of each joint, we can compute the position of the markers and joints in the global coordinate system. Draw it on paper to check my results :)

In [2]:
# Example - 2nd joint of the arm at p_rotated, with a segment length of 1.0 and additional rotation
p1 = torch.tensor([1,0], dtype=torch.float32)  # Position of the second joint relative to the first joint
angle1 = torch.tensor(np.pi / 4, dtype=torch.float32)  # Additional rotation of 45 degrees in radians
# Rotation matrix for the second joint
rotation_matrix1 = torch.tensor([
    [torch.cos(angle1), -torch.sin(angle1)],
    [torch.sin(angle1), torch.cos(angle1)]
], dtype=torch.float32)
# Compute the position of the second joint in global coordinates
p1_global = rotation_matrix @ (rotation_matrix1 @ p1) + p_rotated
print("Position of the second joint in global coordinates:", torch.round(p1_global, decimals=4))

Position of the second joint in global coordinates: tensor([-0.7071,  1.7071])


## 2.2 Forward Kinematics

Now, we want to iterate over all joints of the human body model and apply the rotations to each joint based on the provided angles. Implement the function `forward_kinematics` in `fk/forward_kinematics.py`.
The function should take the following inputs:
- 'q': A ndarray of generalized coordinates (joint angles) of shape (N,).
- 'key': A list of joint names corresponding to the angles in 'q'.
- 'kintree': A dictionary of the kinematic tree as in `model/kintree.py`.

The function should return:
- 'joints': A dictionary where each key is a joint name and the value is a ndarray of shape (3,) representing the 3D position of the joint after applying the forward kinematics.
- 'markers': A dictionary where each key is a marker name and the value is a ndarray of shape (3,) representing the 3D position of the marker after applying the forward kinematics.

In [3]:
# Set up the data
from model.kintree import get_model_dictionary
import pandas as pd

model_dict = get_model_dictionary()

from fk.forward_kinematics import forward_kinematics

q_all = pd.read_csv('data/angles_clean.csv')
marker_data = pd.read_csv('data/markers.csv')
q = q_all.iloc[0,1:] # Exclude time column, 
markers = marker_data.iloc[0,2:] # Exclude time and frame column
key = list(q_all.columns[1:]) # Exclude time column

**To Do:**

Now implement the function `forward_kinematics` in `fk/forward_kinematics.py`.

In [4]:
from utils import evaluate_marker_error, visualize_markers
joint_positions, marker_positions = forward_kinematics(q.values, key, model_dict) # <-- Your implementation here
print("Frame Marker Error (RMSE):", evaluate_marker_error(marker_positions, markers)*1e3, "mm") 
visualize_markers(marker_positions, markers) # Visualize current frame

TypeError: cannot unpack non-iterable NoneType object

Even with a correct forward kinematics implementation, the marker error might still be quite high (> 30 mm). This is because the markers are experimental data (i.e. noisy) and the model is a simplified version of the actual scaled OpenSim model, where some joints have been removed or simplified. For frame 3000, my marker error is 31.46 mm. If your error is substantially higher, please check your implementation again.

At this point, all tests in `pytest tests.py` should pass, only the task of plotting a skeleton stick figure is still open. 

## 2.5 Plotting the Human Body Model

**To Do:**

Implement the function `get_connections` in `fk/forward_kinematics.py` that returns a list of tuples representing the connections between joints for plotting the skeleton. Each tuple should contain two joint names that are connected.



In [ ]:
from fk.forward_kinematics import get_connections
joint_positions, marker_positions = forward_kinematics(q, key, model_dict)
connections = get_connections(model_dict, joint_positions) # <-- Your implementation here

If the function is implemented correctly, you should be able to visualize the skeleton as follows (it may not be beautiful though):

In [ ]:
# Plot skeleton in 2D (sagittal plane)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(6, 8))

# Plot joints
for _joint in joint_positions.values():
    ax.scatter(_joint[0], _joint[1], c='gray', s=20)

# Plot bones
for _bone in connections:
    ax.plot([_bone[0][0], _bone[1][0]], [_bone[0][1], _bone[1][1]], c='k', linewidth=1.5)

# Plot markers
for marker in marker_positions.values():
    ax.scatter(marker[0], marker[1], c='purple', s=8)

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=6, label='Joints'),
    Line2D([0], [0], color='k', linewidth=1.5, label='Bones'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='purple', markersize=5, label='Markers'),
]

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('2D Skeleton View')
ax.set_aspect('equal', adjustable='box')
ax.legend(handles=legend_handles, loc='upper right', frameon=False)
ax.grid(False)
plt.show()